# Load Data

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import category_encoders as ce

excel_file = "Crash Risk Data (1).xls"

# Load the data
df = pd.read_excel(excel_file)
df.head()

,gvkey,fyear,sic,big4,modified,lnmarket_val,lntotal_assets,asset_growth,liability_growth,sales_growth,...,lit,z_score,netfilesize,n_words_den3,n_negative_den3,n_positive_den3,n_uncertainty_den3,n_litigious_den3,crash_new320Plus,ncskew_new_plus
0,10884,1993,100,1,0,6.324117,7.915988,-0.048556,-0.030367,-0.069889,...,0,1.341682,450656,60503,563,226,405,1883,0,-0.760525
1,10390,1994,100,1,1,5.025090,3.804883,-0.042524,-0.235625,-0.082085,...,0,13.269255,190514,20653,244,108,180,457,0,-0.774498
2,8596,1994,100,1,1,7.898852,7.133631,0.026246,-0.063346,0.100678,...,0,7.983232,91626,10501,88,133,104,63,0,0.290179
3,2812,1994,100,0,0,7.221101,8.255486,0.136002,0.220105,0.119820,...,0,1.861746,415553,51758,584,321,520,1466,0,-0.855944
4,14881,1994,100,1,1,5.130932,5.765191,0.008217,-0.024667,0.097770,...,0,2.181140,362277,48629,659,430,281,1499,0,-0.140956


# Transform Data

In [7]:
# Split to train test (we do not use "ncskew_new_plus")
X = df[['gvkey', 'fyear', 'sic', 'big4', 'modified', 'lnmarket_val',
       'lntotal_assets', 'asset_growth', 'liability_growth', 'sales_growth',
       'mtb_at', 'roa2', 'lev_mkt', 'age', 'lit', 'z_score', 'netfilesize',
       'n_words_den3', 'n_negative_den3', 'n_positive_den3',
       'n_uncertainty_den3', 'n_litigious_den3']]
y = df['crash_new320Plus']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Get validation set
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

In [8]:
counts = X_train['sic'].value_counts()
rare_categories = counts[counts < 11].index
print(rare_categories)
# Create an 'Other' category for the rare categories.
X_train['sic'] = X_train['sic'].apply(
    lambda x: 'Other' if x in rare_categories else x
)
X_val['sic'] = X_val['sic'].apply(
    lambda x: 'Other' if x in rare_categories else x
)
# Create a mean embedding for sic.
enc = ce.TargetEncoder(cols=['sic']).fit(X_train, y_train)
X_train = enc.transform(X_train)
X_val = enc.transform(X_val)

Index([6153, 6141, 6513, 6726, 6399, 8351, 7385, 5180, 8111, 3250, 6510, 5734,
       3360, 5094, 3260, 3334, 4822, 3433, 3910, 3444, 5960, 7600, 4013, 2340,
       6159, 1221, 2600, 4610,  200, 6797, 5082, 6162, 3524, 7384, 2673, 7000,
       7320, 3822, 4961, 4220, 5099, 3695, 5020, 2092],
      dtype='int64', name='sic')


In [9]:
# Standardise
scaler = StandardScaler()

categories_to_standardise = ['gvkey', 'fyear', 'sic', 'lnmarket_val',
       'lntotal_assets', 'asset_growth', 'liability_growth', 'sales_growth',
       'mtb_at', 'roa2', 'lev_mkt', 'age', 'z_score', 'netfilesize',
       'n_words_den3', 'n_negative_den3', 'n_positive_den3',
       'n_uncertainty_den3', 'n_litigious_den3']

X_train[categories_to_standardise] = scaler.fit_transform(X_train[categories_to_standardise])
X_val[categories_to_standardise] = scaler.transform(X_val[categories_to_standardise])


In [10]:
# Upsample the minority class
from sklearn.utils import resample

train_data = pd.concat([X_train, y_train], axis=1)

df_majority = train_data[train_data.crash_new320Plus == 0]
df_minority = train_data[train_data.crash_new320Plus == 1]

df_minority_upsampled = resample(df_minority,
                                 replace=True,           
                                 n_samples=len(df_majority),  
                                 random_state=42)  
      
df_upsampled = pd.concat([df_majority, df_minority_upsampled])

X_train = df_upsampled.drop('crash_new320Plus', axis=1)
y_train = df_upsampled['crash_new320Plus']

In [11]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve, auc

def evaluate(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    conf = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = conf.ravel()
    sensitivity = TP / (TP + FN)
    specificity = TN / (TN + FP)
    f1 = f1_score(y_true, y_pred, average=None)
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    area = auc(fpr, tpr)
    print(f"The accuracy is: {acc:.4f}")
    print(f"The F1 score is: {np.round(f1,4)}")
    print(f"AUC is: {area:.4f}")
    print(f"The sensitivity is {sensitivity:.4f}")
    print(f"The specificity is {specificity:.4f}")
    print(f"The confusion matrix is: \n {conf}")

# Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=42, solver='newton-cholesky', C=0.01).fit(X_train, y_train)
clf.fit(X_train, y_train)
clf_preds = clf.predict(X_val)
evaluate(y_val, clf_preds)

The accuracy is: 0.5928
The F1 score is: [0.7058 0.3389]
AUC is: 0.5876
The sensitivity is 0.5794
The specificity is 0.5958
The confusion matrix is: 
 [[4830 3277]
 [ 749 1032]]
